In [1]:
# Install and set up
!pip install scikit-learn

import sys
sys.path.append('./Property-awareness-Representation-Learning')

# Core libraries
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Data and utilities
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from utils import load_mat, CombinedDataset, loss_function


In [9]:
# Load the .mat files
shape_data = load_mat("ShapeSpace.mat")["ShapeSpace"]         # (50, 50, 248396)
property_data = load_mat("PropertySpace.mat")["PropertySpace"]  # (5, 248396)

# Fix shape: (50, 50, 248396) → (248396, 1, 50, 50)
shape_data = np.transpose(shape_data, ( 0 , 2 , 1))               # (248396, 50, 50)
shape_data = shape_data[:, np.newaxis, :, :]                   # (248396, 1, 50, 50)
property_data = property_data.T                                # (248396, 5)

# Convert to float and wrap into a dataset
dataset = CombinedDataset(shape_data.astype(np.float32), property_data.astype(np.float32))
# You can add line to select dataset "dataset = dataset[1000]"
# Train-test split
train_dataset, test_dataset = train_test_split(dataset, test_size=0.1, random_state=42)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, drop_last=True)

# Optional: Confirm shape
print("Shape data:", shape_data.shape)        # (248396, 1, 50, 50)
print("Property data:", property_data.shape)  # (248396, 5)


Shape data: (248396, 1, 50, 50)
Property data: (248396, 5)


In [10]:
import torch.nn.functional as F

class CNNPropertyPredictor(nn.Module):
    def __init__(self):
        super(CNNPropertyPredictor, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)   # Output: (16, 50, 50) ##important 3 parameters: stride number --> out size = ninput size
        self.pool = nn.MaxPool2d(2, 2)                            # Output: (16, 25, 25)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1) # Output: (32, 25, 25)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 25 * 25, 128)
        self.fc2 = nn.Linear(128, 5)  # Output: 5 property predictions

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # (B, 16, 25, 25)
        x = F.relu(self.conv2(x))             # (B, 32, 25, 25)
        x = self.flatten(x)                   # (B, 32*25*25)
        x = F.relu(self.fc1(x))               # (B, 128)
        x = self.fc2(x)                       # (B, 5)
        return x


In [11]:
model = CNNPropertyPredictor()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_function = nn.MSELoss()


In [12]:
epochs = 10  # You can adjust
model.train()

for epoch in range(epochs):
    total_loss = 0
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        y_pred = model(x)
        loss = loss_function(y_pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")


Epoch 1/10, Loss: 0.0029
Epoch 2/10, Loss: 0.0011
Epoch 3/10, Loss: 0.0009
Epoch 4/10, Loss: 0.0007
Epoch 5/10, Loss: 0.0006
Epoch 6/10, Loss: 0.0006
Epoch 7/10, Loss: 0.0006
Epoch 8/10, Loss: 0.0005
Epoch 9/10, Loss: 0.0004
Epoch 10/10, Loss: 0.0004


In [13]:
# Testing the CNN Prediction Model
model.eval()  # Set model to evaluation mode (no dropout, no gradients)
test_loss = 0

with torch.no_grad():  # Don’t calculate gradients while testing
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        y_pred = model(x)  # Make prediction
        loss = loss_function(y_pred, y)  # Calculate how far off the prediction is
        test_loss += loss.item()  # Accumulate total loss

# Calculate average test loss
avg_test_loss = test_loss / len(test_loader)
print(f"\n Average test loss: {avg_test_loss:.4f}")



 Average test loss: 0.0004


In [16]:
torch.save(model.state_dict(), "cnn_model.pt")
